# 그래프 구성

공시(DART)·뉴스 최종본, 이름 통일, 제품 사전, WI26 업종, 토스증권 테마를 모아 **노드 표와 관계 표(CSV)를 만들고 Neo4j에 통째로 넣는다**(`graphDB.md` 6장).

- 표는 원본 결과에서 언제든 다시 만든다. 규칙이나 사전이 바뀌면 이 노트북을 다시 실행해 Neo4j를 비우고 다시 넣는다.
- 노드마다 고정 ID를 둔다. 같은 대상은 몇 번 나와도 노드 하나다.
- 같은 두 노드 사이의 같은 종류 관계는 **하나로 합치고**, 출처(DART·뉴스), 근거 문장과 그 출처(기사 ID 또는 공시 접수번호), 상태 등을 목록으로 붙인다.
- **기사와 공시는 노드로 두지 않는다.** 관계의 근거로만 쓰고, 출처 목록 표(`source/graph/sources/`)에서 제목·링크를 찾는다. 사용자에게 "이 관계는 이 공시·기사에서 나왔다"고 보여 줄 때 쓴다.
- 서버나 LLM에 요청하지 않는다. Neo4j는 이 맥의 로컬 Docker(`neo4j-graph`)에 넣는다.

**입력**

| 파일 | 쓰는 것 |
|---|---|
| `source/graph/companies.csv`, `company_mentions.csv`, `company_ambiguous.csv` | 기업 노드와 원문 언급 → 기업 노드 대응(`name_unifier`) |
| `source/graph/products_final.csv`, `product_mentions_final.csv`, `product_hierarchy.csv` | 제품군·제품·후보 하위 제품 노드, 언급 → 제품 대응(여러 품목 언급은 여러 제품), 계층(`product_apply`) |
| `source/graph/brands_final.csv`, `brand_products.csv` | 브랜드 노드(규칙 사전 + 종류)와 브랜드 → 제품(LLM 3단계, `product_apply`). 언급 → 브랜드는 `product_mentions_final.csv`에 함께 있다 |
| `source/news/final/news_final.json` | 기사, 개체, 관계, 사건, 기사 업종(`news_cleaner`) |
| `source/dart/business/final/dart_business_v5.json` | 제품, 원재료·매입처, 고객, 경쟁사, 계약, 기타 관계, 브랜드 |
| `source/dart/parsed/subsidiaries.csv`, `associates.csv`, `investments.csv`, `shareholders.csv` | 지분 관계 |
| `source/dart/parsed/affiliates.csv`, `affiliate_groups.csv` | 사업보고서 계열회사 현황: 그룹 소속(`dart_parser` 10절) |
| `../market_data/sector.json` | 상장사 2,655개와 WI26 업종(서비스 대상 후보만 쓴다) |
| `../market_data/toss_theme.json` | 상장사별 토스증권 테마(상위 태그 125개) |
| `source/targets.csv` | 관계 추출 대상 1,067개 |

**출력**: `source/graph/neo4j/`의 노드 표(`nodes_<라벨>.csv`)와 관계 표(`rels_<관계>.csv`), 출처 목록 `source/graph/sources/articles.csv`·`filings.csv`, 그리고 Neo4j.

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `name_unifier` → `product_dictionary` → `product_llm` → `product_apply` → **`graph_db`** (마지막 단계) |
| 외부 요청·비용 | 없음. 로컬 Docker의 Neo4j에만 접속한다(비밀번호는 `.env`) |
| 다시 실행 | 언제든 된다. `LOAD=True`면 Neo4j를 비우고 처음부터 다시 넣는다 |
| 멈추는 경우 | Neo4j가 꺼져 있으면 표만 만들고 접속 단계에서 멈춘다 |
| 처리 순서 | 설정 → 대응표 → 상장사·업종·테마 → 기업 ID·그룹 → 노드 → 소속 관계 → DART 지분 → DART 사업 → 뉴스 관계 → 관계 합치기 → 표·출처 저장 → Neo4j 넣기 → 확인 |

## 1. 설정

In [1]:
# ── 설정: 폴더와 상수 ──
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'
GRAPH = ROOT / 'source' / 'graph'
PARSED = ROOT / 'source' / 'dart' / 'parsed'
MARKET = ROOT.parent / 'market_data'
OUT = GRAPH / 'neo4j'
OUT.mkdir(parents=True, exist_ok=True)
SRC = GRAPH / 'sources'
SRC.mkdir(parents=True, exist_ok=True)

LOAD = True               # True면 표를 만든 뒤 Neo4j를 비우고 다시 넣는다
BATCH = 5_000             # Neo4j에 한 번에 보내는 줄 수
MAX_ARTICLES = 50         # 관계 하나에 남기는 기사 ID 수(관계의 기사 수 n_news는 전부 센다)
MAX_EVIDENCE = 3          # 관계 하나에 남기는 근거 문장 수(최근 것부터, 출처마다 하나)

## 2. 노드와 관계의 종류

**노드** (모든 노드에 공통 라벨 `Node`와 고정 ID `id`, 표시 이름 `name`이 있다)

| 라벨 | ID | 무엇 |
|---|---|---|
| `Company` | `KR:종목코드`, `EN:영어이름`, `KO:한글이름` | 기업. 서비스 대상 후보 1,067개(`targets.csv`, 업종·테마에 잇는다)와 관계 상대 |
| `Group` | `G:samsung` | 그룹 이름만 적힌 언급("Samsung", "포스코", "LG") |
| `Sector` | `WI:WI100` | WI26 업종 26개 |
| `Theme` | `TH:방산` | 토스증권 테마 |
| `Product` | `P:`, `PC:`, `PR:` | 제품·기술·원자재·지표(`level`: 제품 / 후보 하위 제품 `PC:`, 검색어로만 / LLM 결과 전의 임시 제품 `PR:`) |
| `ProductFamily` | `PF:` | 제품군(LLM 제품 사전의 확정 제품군. 계층의 맨 위) |
| `Brand` | `B:` | 상품·브랜드 |
| `Policy`, `Organization`, `Facility`, `Project` | `N:종류:이름` | 뉴스의 정책·규제, 기관, 시설, 프로젝트·계약 |

**관계** (뉴스 역할 15개 + 기타, DART 관계는 같은 이름에 넣는다)

| 관계 | 뜻(방향) | 출처 |
|---|---|---|
| `SUPPLIES` | 공급(파는 쪽 → 사는 쪽) | 뉴스 공급, DART 고객(주요 고객·판매 채널)·매입처·공급 계약 |
| `BUYS` | 구매·사용(쓰는 쪽 → 산 것) | 뉴스 구매·사용, DART 원재료 |
| `PRODUCES` | 생산(만드는 쪽 → 제품) | 뉴스 생산, DART 제품 |
| `WINS_ORDER` | 수주·발주(따낸 쪽 → 발주한 쪽) | 뉴스, DART 고객 중 수주 상대 |
| `INVESTS`, `DIVESTS` | 인수·투자, 매각 | 뉴스 |
| `PARTNERS` | 합작·제휴 | 뉴스, DART 계약(합작·제휴) |
| `LICENSES` | 라이선스(주는 쪽 → 받는 쪽) | 뉴스, DART 계약(라이선스·기술이전, 방향 불명) |
| `COMPETES` | 경쟁 | 뉴스, DART 경쟁사 |
| `REGULATES`, `BUILDS_OPERATES`, `AFFECTS`, `DISPUTES`, `FUNDS`, `POLICY_STANCE`, `OTHER` | 규제·인허가, 시설 건설·운영, 영향(+/−), 소송·분쟁, 자금 지원, 정책 입장, 기타 | 뉴스(OTHER는 DART 계약 기타·기타 관계도) |
| `OWNS` | 지분(가진 쪽 → 대상, 지분율) | DART 종속회사·관계기업·출자·법인 주주 |
| `HAS_BRAND` | 브랜드 | DART 브랜드 |
| `IN_SECTOR`, `HAS_THEME`, `IN_GROUP` | 업종·테마·그룹 소속 | WI26, 토스, 사업보고서 계열회사 현황 |
| `SUBTYPE_OF` | 제품 계층(하위 제품 → 상위 제품 → 제품군, 후보 → 제품). 근거 문장이 없어 경로에서는 "분류"로 보여 준다 | LLM 제품 사전 |
| `OF_PRODUCT` | 브랜드 → 제품(제품을 못 찾았으면 제품군). 계층처럼 "분류"로 보여 준다 | LLM 제품 사전 3단계 |

In [2]:
# ── 대응표: 뉴스 관계 역할 → 관계 종류, 그 밖의 개체 종류 → 노드 라벨 ──
ROLE_TYPE = {'공급': 'SUPPLIES', '구매·사용': 'BUYS', '생산': 'PRODUCES', '수주·발주': 'WINS_ORDER', '인수·투자': 'INVESTS',
             '매각': 'DIVESTS', '합작·제휴': 'PARTNERS', '라이선스': 'LICENSES', '경쟁': 'COMPETES', '규제·인허가': 'REGULATES',
             '시설 건설·운영': 'BUILDS_OPERATES', '영향': 'AFFECTS', '소송·분쟁': 'DISPUTES', '자금 지원': 'FUNDS',
             '정책 입장': 'POLICY_STANCE', '기타': 'OTHER'}
OTHER_LABEL = {'정책·규제': ('Policy', 'policy'), '기관': ('Organization', 'org'), '시설': ('Facility', 'facility'),
               '프로젝트·계약': ('Project', 'project')}

## 3. 상장사, 업종, 테마

- **서비스 대상 후보**는 `source/targets.csv`의 1,067개다(`target=True`). 이 목록에 없는 종목은 서비스 대상이 되지 않는다(graphDB.md 2절). 스팩·리츠·인프라펀드·해외 기업은 이 목록을 만들 때 이미 빠졌다.
- 업종·테마 소속은 후보에만 붙인다. 후보 밖 상장사는 다른 기업과 관계가 있을 때만 일반 기업 노드로 남고, 업종·테마는 속성으로만 둔다.
- 그룹 소속은 공시(계열회사 현황)에 근거한 관계라 그래프에 남은 기업 모두에 붙인다(4절).
- 토스 테마 파일에는 종목코드가 없어 `sector.json`의 종목 이름으로 코드를 찾는다.

In [3]:
# ── 상장사·WI26 업종·토스 테마와 서비스 대상 후보(targets.csv) ──
SEC = json.loads((MARKET / 'sector.json').read_text(encoding='utf-8'))
stocks = SEC['stocks']
targets = pd.read_csv(ROOT / 'source' / 'targets.csv', dtype=str)
TARGET = set(targets['stock_code'])                 # 서비스 대상 후보. 업종·테마·그룹 소속은 이 종목에만 붙인다
name2code = {v['security_name']: k for k, v in stocks.items()}
theme_raw = json.loads((MARKET / 'toss_theme.json').read_text(encoding='utf-8'))
THEME = {}
for x in theme_raw:
    THEME[name2code[x['기업이름']]] = x['상위태그']       # 이름이 모두 맞는다(README 6)
assert TARGET <= set(stocks), '후보가 sector.json에 없다'
print('상장사', len(stocks), '/ 서비스 대상 후보', len(TARGET), '/ 후보 중 테마 있는 종목', sum(1 for c in TARGET if THEME.get(c)))
SECTOR_CODE = {v['sector_name']: k for k, v in SEC['sectors'].items()}

상장사 2655 / 서비스 대상 후보 1067 / 후보 중 테마 있는 종목 1066


## 4. 기업 노드

- 기업 노드는 이름 통일 결과(`companies.csv`)다. 후보 밖 상장사와 이름이 같은 언급은 이름 통일이 이미 `KR:종목코드`로 이었다. 상장사 2,655개 노드를 모두 만든 뒤, 관계가 없는 후보 밖 상장사는 5.3 끝에서 뺀다.
- 한글 이름 노드(`KO:`)가 **두 글자 이하**의 후보 밖 상장사 이름(삼진, 서한, 태양 등)과 같으면, 다른 회사일 수 있어 잇지 않고 `maybe_listed` 속성에만 적는다.
- 그룹 이름만 적힌 언급(이름 통일의 "애매")은 그룹 노드로 잇는다. 이런 그룹 18개(`GROUPS`)는 대표 상장사로 아래 계열회사 그룹과 맞춘다.
- **그룹 소속은 사업보고서 계열회사 현황**(`dart_parser` 10절의 `affiliates.csv`)으로 정한다. 보고 회사와 그 목록의 상장 계열회사를 같은 그룹으로 묶고, 상장 계열회사가 2개 이상인 것을 그룹으로 둔다(대표 그룹은 하나여도 둔다). 기업집단 이름은 표기가 달라(엘지, LG계열) 묶는 데 쓰지 않는다.
- 소속 관계(`IN_GROUP`)는 노드를 거른 뒤(5.3 끝) 그래프에 남은 기업에 잇는다. 비상장 계열회사는 DART 언급 이름이 한 기업 노드만 가리킬 때 잇는다. 근거는 그 사업보고서다.

In [4]:
# ── 기업 ID 정리: 두 글자 이하 상장사 이름 표시(maybe_listed), 그룹 이름 언급을 그룹 노드에 잇는다 ──
CO = pd.read_csv(GRAPH / 'companies.csv', dtype=str).fillna('')
CM = pd.read_csv(GRAPH / 'company_mentions.csv', dtype=str).fillna('')
AMB = pd.read_csv(GRAPH / 'company_ambiguous.csv', dtype=str).fillna('')
norm_ko = lambda s: re.sub(r'\(주\)|㈜|주식회사|\(유\)|유한회사|\s', '', s or '')

# 후보 밖 상장사와 이름이 같은 언급은 name_unifier가 KR:로 이었다. 두 글자 이하 이름만 다른 회사일 수 있어 잇지 않고 표시한다
listed_by_name = {norm_ko(v['security_name']): k for k, v in stocks.items() if k not in TARGET}
maybe = {}
for r in CO[CO['entity_id'].str.startswith('KO:')].itertuples():
    k = norm_ko(r.name_ko)
    if k in listed_by_name and len(k) <= 2:
        maybe[r.entity_id] = 'KR:' + listed_by_name[k]
print('두 글자 이하 상장사 이름과 같은 한글 노드(잇지 않고 표시)', len(maybe))

# 그룹: 그룹 이름 언급("Samsung", "현대")을 잇는 표기와 대표 상장사. 소속은 아래에서 사업보고서 계열회사 현황으로 정한다
# 그룹 ID → (표시 이름, 그룹 이름 표기, 대표 상장사 종목코드). 해외 그룹은 대표 상장사가 없다
GROUPS = {
    'samsung': ('삼성', ['samsung', '삼성'], '005930'),
    'lg': ('LG', ['lg', 'lg corp', 'lg corporation', '엘지'], '003550'),
    'sk': ('SK', ['sk', 'sk corp', 'sk inc', '에스케이'], '034730'),
    'hyundai': ('현대(현대차그룹)', ['hyundai', '현대'], '005380'),
    'hd hyundai': ('HD현대', ['hd hyundai', 'hd현대'], '267250'),
    'lotte': ('롯데', ['lotte', '롯데'], '004990'),
    'hanwha': ('한화', ['hanwha', '한화'], '000880'),
    'doosan': ('두산', ['doosan', '두산'], '000150'),
    'gs': ('GS', ['gs', 'gs corp', '지에스'], '078930'),
    'cj': ('CJ', ['cj', 'cj corp', '씨제이'], '001040'),
    'posco': ('포스코', ['posco', '포스코'], '005490'),
    'hyosung': ('효성', ['hyosung', '효성'], '004800'),
    'kolon': ('코오롱', ['kolon', '코오롱'], '002020'),
    'ls': ('LS', ['ls', 'ls corp', '엘에스'], '006260'),
    'kumho': ('금호', ['kumho', '금호'], '011780'),
    'hanjin': ('한진', ['hanjin', '한진'], '180640'),
    'shinsegae': ('신세계', ['shinsegae', '신세계'], '004170'),
    'kt': ('KT', ['kt', 'kt corp', '케이티'], '030200'),
    'mitsubishi': ('Mitsubishi', ['mitsubishi', '미쓰비시'], None),
    'mitsui': ('Mitsui', ['mitsui', '미쓰이'], None),
    'sumitomo': ('Sumitomo', ['sumitomo', '스미토모'], None),
    'marubeni': ('Marubeni', ['marubeni', '마루베니'], None),
}
assert all(a in stocks for _, _, a in GROUPS.values() if a), '대표 상장사가 sector.json에 없다'
def group_key(name):
    """그룹 이름 언급("Samsung Group" 등)을 그룹 ID(G:…)로 바꾼다. 그룹이 아니면 None."""
    k = re.sub(r'\(주\)|㈜|주식회사', '', name or '').lower()
    k = re.sub(r'\b(co|corp|corporation|inc|ltd|group|holdings?)\b\.?', '', k)
    k = re.sub(r'[^0-9a-z가-힣 ]', ' ', k)
    k = re.sub(r'\s+', ' ', k).strip()
    for g, (_, aliases, _) in GROUPS.items():
        if k in aliases:
            return 'G:' + g
    return None
AMB['group'] = AMB['name'].map(group_key)
print('그룹 이름 언급', len(AMB), '/ 그룹에 이음', AMB['group'].notna().sum(), '/ 못 이음', AMB.loc[AMB['group'].isna(), 'name'].value_counts().head(8).to_dict())

# 그룹 소속: 사업보고서 계열회사 현황(dart_parser 10절). 보고 회사와 그 목록의 상장 계열회사를 같은 그룹으로 묶는다(union-find).
# 기업집단 이름은 회사마다 표기가 달라(엘지, LG계열) 묶는 데 쓰지 않고 표시 이름으로만 쓴다. 상장 계열회사가 2개 이상인 것만 그룹으로 둔다
# (그룹 이름 언급을 받는 GROUPS의 대표 상장사가 든 그룹은 상장 계열회사가 하나여도 둔다: 금호석유화학)
AFF = pd.read_csv(PARSED / 'affiliates.csv', dtype=str).fillna('')
AFG = pd.read_csv(PARSED / 'affiliate_groups.csv', dtype=str).fillna('')
gparent = {}


def gfind(x):
    """상장사 x가 속한 그룹 덩어리의 대표."""
    gparent.setdefault(x, x)
    while gparent[x] != x:
        gparent[x] = gparent[gparent[x]]
        x = gparent[x]
    return x


for r in AFF[AFF['affiliate_stock_code'] != ''].itertuples():
    gparent[gfind(r.affiliate_stock_code)] = gfind(r.stock_code)
comp = defaultdict(set)
for c in list(gparent):
    comp[gfind(c)].add(c)
clean_group = lambda n: re.sub(r'\(주\)|㈜|주식회사|그룹|계열|및\s*관계사|\([^)]*\)|\s', '', n or '')
anchor = {a: g for g, (_, _, a) in GROUPS.items() if a}
GROUP_OF, GROUP_NAMES = {}, {}            # 종목코드 → 그룹 ID, 그룹 ID → 보고서에 적힌 기업집단 이름들
for c in anchor:
    gfind(c)
comp = defaultdict(set)
for c in list(gparent):
    comp[gfind(c)].add(c)
for ms in comp.values():
    if len(ms) < 2 and not ms & set(anchor):
        continue
    names = Counter(n for n in AFG[AFG['stock_code'].isin(ms)]['group'] if n)
    known = sorted({anchor[c] for c in ms if c in anchor})
    if known:
        gid = 'G:' + known[0]
    else:
        base = clean_group(names.most_common(1)[0][0]) if names else stocks[min(ms)]['security_name']
        gid = 'G:' + base
        if gid in GROUP_NAMES:                # 다른 그룹과 이름이 겹치면 대표 종목 이름을 붙인다
            gid = f"G:{base}({stocks[min(ms)]['security_name']})"
    GROUP_NAMES[gid] = [n for n, _ in names.most_common(5)]
    for c in ms:
        GROUP_OF[c] = gid
print('계열회사 현황으로 정한 그룹', len(GROUP_NAMES), '/ 소속 상장사', len(GROUP_OF),
      '/ 대표 상장사로 이름을 붙인 그룹', sum(1 for g in GROUP_NAMES if g[2:] in GROUPS))

두 글자 이하 상장사 이름과 같은 한글 노드(잇지 않고 표시) 26
그룹 이름 언급 336 / 그룹에 이음 334 / 못 이음 {'미쯔비시': 1, 'LG복지몰': 1}
계열회사 현황으로 정한 그룹 236 / 소속 상장사 814 / 대표 상장사로 이름을 붙인 그룹 18


In [5]:
# ── 노드 만들기: 기업, 그룹, 업종, 테마, 제품, 브랜드 ──
nodes = defaultdict(dict)          # 라벨 → {id: 속성}

def add_node(label, id_, **props):
    """노드를 더한다. 이미 있으면 비어 있는 속성만 채운다."""
    cur = nodes[label].get(id_)
    if cur is None:
        nodes[label][id_] = {'id': id_, **props}
    else:
        for k, v in props.items():
            if cur.get(k) in (None, '', []) and v not in (None, '', []):
                cur[k] = v

split = lambda s: [x for x in (s or '').split(' | ') if x]
# 이름 통일의 기업 노드
for r in CO.itertuples():
    add_node('Company', r.entity_id, name=r.name_ko or r.name_en, name_en=r.name_en, name_ko=r.name_ko, country=r.country,
             other_names=split(r.other_names), n_news=int(r.n_news or 0), n_dart=int(r.n_dart or 0),
             maybe_listed=maybe.get(r.entity_id, ''))
# 상장사 2,655개는 언급이 없어도 노드로 두고 시장·업종·테마를 붙인다
for c, v in stocks.items():
    i = 'KR:' + c
    add_node('Company', i, name=v['security_name'], name_ko=v['security_name'], country='대한민국')
    n = nodes['Company'][i]
    n.update(listed=True, stock_code=c, market=v['market'], target=c in TARGET,
             sector=v['wi26']['sector_name'], themes=THEME.get(c, []), name=v['security_name'])
for g, (disp, aliases, _) in GROUPS.items():
    add_node('Group', 'G:' + g, name=disp, aliases=aliases)
for gid, names in GROUP_NAMES.items():         # 계열회사 현황으로 정한 그룹. official_names는 보고서에 적힌 기업집단 이름
    add_node('Group', gid, name=gid[2:], official_names=names)
    nodes['Group'][gid]['official_names'] = names
for k, v in SEC['sectors'].items():
    add_node('Sector', 'WI:' + k, name=v['sector_name'], code=k)
for t in sorted({t for ts in THEME.values() for t in ts}):
    add_node('Theme', 'TH:' + t, name=t)

# 제품(LLM 제품 사전 적용 결과: 제품군·제품·후보·임시)과 브랜드(규칙 제품 사전)
PRODUCTS = pd.read_csv(GRAPH / 'products_final.csv', dtype=str).fillna('')
for r in PRODUCTS.itertuples():
    add_node('ProductFamily' if r.level == '제품군' else 'Product', r.id, name=r.name_ko or r.name_en, name_en=r.name_en, name_ko=r.name_ko,
             level=r.level, family=r.family, kind=r.kind, category=r.category, specs=split(r.specs), search_terms=split(r.search_terms),
             sectors=split(r.sectors), generic=r.generic == 'True', candidate=r.candidate == 'True', pending_llm=r.pending_llm == 'True',
             intermediate=r.intermediate == 'True', n_kr_makers=int(r.n_kr_makers or 0), n_news=int(r.n_news or 0), n_mentions=int(r.n_mentions or 0))
for r in pd.read_csv(GRAPH / 'brands_final.csv', dtype=str).fillna('').itertuples():
    add_node('Brand', r.id, name=r.name_ko or r.name_en, name_en=r.name_en, name_ko=r.name_ko, category=r.category, brand_kind=r.brand_kind,
             search_terms=split(r.search_terms), sectors=split(r.sectors), generic=r.generic == 'True',
             n_kr_makers=int(r.n_kr_makers or 0), n_news=int(r.n_news or 0), n_mentions=int(r.n_mentions or 0))
print({k: len(v) for k, v in nodes.items()})

{'Company': 41906, 'Group': 240, 'Sector': 26, 'Theme': 125, 'Product': 15496, 'ProductFamily': 273, 'Brand': 5596}


## 5. 관계 모으기

관계는 하나씩 `add_rel`로 모으고, 끝에 (시작, 종류, 끝)이 같은 것을 하나로 합친다.

In [6]:
# ── 소속 관계: 업종(IN_SECTOR), 테마(HAS_THEME). 그룹(IN_GROUP)은 5.3 끝에서 ──
rels = defaultdict(list)           # (시작, 종류, 끝) → 낱개 관계 목록

def add_rel(s, t, e, **p):
    """낱개 관계를 더한다. 같은 (시작, 종류, 끝)은 나중에 하나로 합친다. 자기 자신과의 관계는 버린다."""
    if s and e and s != e:
        rels[(s, t, e)].append(p)

# 업종·테마 소속: 서비스 대상 후보에만. 그룹 소속은 공시에 근거한 관계라 5.3 끝에서 그래프에 있는 기업 모두에 잇는다
for c in sorted(TARGET):
    v = stocks[c]
    add_rel('KR:' + c, 'IN_SECTOR', 'WI:' + v['wi26']['sector_code'], source='wi26')
    for t in THEME.get(c, []):
        add_rel('KR:' + c, 'HAS_THEME', 'TH:' + t, source='toss')
print('소속 관계', Counter(k[1] for k in rels))

소속 관계 Counter({'HAS_THEME': 2153, 'IN_SECTOR': 1067})


### 5.1 DART 지분

- 종속회사·관계기업·출자 대상: 상장사 → 상대. 법인 주주: 주주 → 상장사.
- 같은 두 회사가 여러 표에 나오면(관계기업이면서 출자 대상) 관계 하나로 합치고 `kinds`에 모두 적는다. 지분율은 가장 큰 값.
- 근거는 공시 접수번호(`D:접수번호`)와 표 이름으로 남긴다(표 한 줄이라 문장 근거는 없다).
- 출자 대상의 목적(경영참여·단순투자 등)은 `notes`에 남긴다. 단순투자(예: 지주사가 가진 Nvidia 주식)는 사업 관계가 아니므로 경로를 찾을 때 거를 수 있게 한다.

In [7]:
# ── DART 지분 관계(OWNS): 종속회사·관계기업·출자 대상·법인 주주. 공시는 출처 목록에 담는다 ──
cm_dart = {(r.source, r.doc, r.field, r.name): r.entity_id for r in CM.itertuples() if r.source != 'news'}
amb_dart = {(r.source, r.doc, r.field, r.name): r.group for r in AMB.itertuples() if r.source != 'news' and r.group}
def dart_co(source, doc, field, name):
    """DART 언급을 그래프 기업 ID(또는 그룹 ID)로 바꾼다."""
    i = cm_dart.get((source, doc, field, name)) or amb_dart.get((source, doc, field, name))
    return i or None

FILINGS = {}                       # 접수번호 → 공시 정보(출처 목록)
name_of = {v['security_name'] for v in stocks.values()}
def filing(rcept, code_, report, base_date):
    """공시 하나를 출처 목록에 더한다(ref = D:접수번호, DART 링크)."""
    if rcept and rcept not in FILINGS:
        FILINGS[rcept] = dict(ref='D:' + rcept, rcept_no=rcept, stock_code=code_,
                              corp_name=stocks.get(code_, {}).get('security_name', ''), report=report, base_date=base_date,
                              url=f'https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept}')

def num(x):
    """숫자로 바꾼다. 안 되면 None."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

miss = Counter()
for f, field in [('subsidiaries', '종속회사'), ('associates', '관계기업'), ('investments', '출자 대상')]:
    t = pd.read_csv(PARSED / f'{f}.csv', dtype=str).fillna('')
    for r in t.itertuples():
        e = dart_co('dart_지분', r.stock_code, field, r.name)
        if not e:
            miss[field] += 1
            continue
        # 근거는 "보고서 표: 회사 이름" 문장, 출처는 그 공시
        add_rel('KR:' + r.stock_code, 'OWNS', e, source='dart', doc=r.stock_code, kind=field,
                ratio=num(getattr(r, 'ratio', None)), note=getattr(r, 'purpose_group', ''),
                ref='D:' + r.rcept_no, date=r.base_date, evidence=f'{r.report} {field} 표: {r.name}')
        filing(r.rcept_no, r.stock_code, r.report, r.base_date)
sh = pd.read_csv(PARSED / 'shareholders.csv', dtype=str).fillna('')
for r in sh[sh['is_corp'] == 'True'].itertuples():
    s = dart_co('dart_지분', r.stock_code, '법인 주주', r.name)
    if not s:
        miss['법인 주주'] += 1
        continue
    # 법인 주주는 방향이 반대(주주 → 상장사)
    add_rel(s, 'OWNS', 'KR:' + r.stock_code, source='dart', doc=r.stock_code, kind='법인 주주', ratio=num(r.ratio),
            note=r.relation, ref='D:' + r.rcept_no, date=r.base_date, evidence=f'{r.report} 최대주주 현황: {r.name} {r.ratio}%')
    filing(r.rcept_no, r.stock_code, r.report, r.base_date)
print('지분 관계', sum(1 for k in rels if k[1] == 'OWNS'), '/ 상대를 못 찾아 뺀 것', dict(miss))

지분 관계 39777 / 상대를 못 찾아 뺀 것 {}


### 5.2 DART 사업의 내용

| DART 항목 | 관계 |
|---|---|
| 제품 | 상장사 `PRODUCES` 제품 |
| 원재료 | 상장사 `BUYS` 원재료(제품 노드), 매입처 `SUPPLIES` 상장사(매입 품목은 `products`) |
| 고객: 주요 고객·판매 채널 / 수주 상대 | 상장사 `SUPPLIES` 고객 / 상장사 `WINS_ORDER` 발주처 |
| 경쟁사 | 상장사 `COMPETES` 경쟁사 |
| 계약: 공급 / 라이선스·기술이전 / 합작·제휴 / 기타 | `SUPPLIES` / `LICENSES`(방향 불명) / `PARTNERS` / `OTHER` |
| 기타 관계 | `OTHER` |
| 브랜드 | 상장사 `HAS_BRAND` 브랜드(소유: 자사·불명·고객사) |

In [8]:
# ── DART 사업의 내용 관계: 제품·원재료·매입처·고객·경쟁사·계약·기타 관계·브랜드 ──
BIZ = json.loads((ROOT / 'source' / 'dart' / 'business' / 'final' / 'dart_business_v5.json').read_text(encoding='utf-8'))['companies']
PM = pd.read_csv(GRAPH / 'product_mentions_final.csv', dtype=str).fillna('')
# 언급 → 제품 ID 목록. 여러 품목이 적힌 언급("레미콘, 이형철근")은 제품이 여러 개라 관계도 제품마다 생긴다
pm_dart, pm_news = defaultdict(list), defaultdict(list)
for r in PM.itertuples():
    k, d = ((r.doc, r.field, r.name), pm_dart) if r.source == 'dart' else ((r.doc, r.ref), pm_news)
    if r.product_id not in d[k]:
        d[k].append(r.product_id)

# 입력 파일 머리말의 접수번호·기준일·보고서
# 사업보고서 접수번호·기준일은 LLM 입력 파일 머리말에서 읽는다
BIZ_REPORT = {}
for c in BIZ:
    head = (ROOT / 'source' / 'dart' / 'business' / 'inputs' / f'{c}.txt').read_text(encoding='utf-8')[:400]
    m1, m2 = re.search(r'접수번호: (\d+)', head), re.search(r'기준일: ([\d-]+)', head)
    m3 = re.search(r'\) (.+보고서 \([\d.]+\))', head.splitlines()[0])
    BIZ_REPORT[c] = (m1.group(1) if m1 else '', m2.group(1) if m2 else '', m3.group(1) if m3 else '사업보고서')
    filing(BIZ_REPORT[c][0], c, BIZ_REPORT[c][2], BIZ_REPORT[c][1])
print('사업보고서 접수번호 없는 회사', sum(1 for v in BIZ_REPORT.values() if not v[0]))

CONTRACT_TYPE = {'공급': 'SUPPLIES', '라이선스': 'LICENSES', '기술이전': 'LICENSES', '합작·제휴': 'PARTNERS', '기타': 'OTHER'}
miss = Counter()
for c, d in BIZ.items():
    me = 'KR:' + c
    ev = {e['id']: e['text'] for e in d['evidence']}
    # 모든 DART 사업 관계의 공통 속성: 근거 문장, 출처(공시), 날짜
    base = lambda x: dict(source='dart', doc=c, evidence=ev.get(x.get('evidence_id'), ''),
                          ref='D:' + BIZ_REPORT[c][0], date=BIZ_REPORT[c][1])
    for p in d['products']:
        ps = pm_dart.get((c, '제품', p['name']))
        for node in ps or []:
            add_rel(me, 'PRODUCES', node, **base(p), note=p.get('segment', ''), stage=p.get('stage', ''))
        if not ps:
            miss['제품'] += 1
    for m in d['materials']:
        ps = pm_dart.get((c, '원재료', m['item']))
        for node in ps or []:
            add_rel(me, 'BUYS', node, **base(m), note=m.get('use', ''))
        if not ps:
            miss['원재료'] += 1
        for s in m['suppliers']:
            e = dart_co('dart_사업', c, '매입처', s['name'])
            if e:
                # 매입처 → 이 회사로 공급(무엇을 공급했는지는 product, 여러 품목이면 제품마다)
                for node in ps or ['']:
                    add_rel(e, 'SUPPLIES', me, **base(m), product=node, note=s.get('note', ''))
            else:
                miss['매입처'] += 1
    for x in d['customers']:
        e = dart_co('dart_사업', c, '고객', x['name'])
        if e:
            add_rel(me, 'WINS_ORDER' if x['kind'] == '수주 상대' else 'SUPPLIES', e, **base(x), kind=x['kind'], note=x.get('product', ''))
        else:
            miss['고객'] += 1
    for x in d['competitors']:
        e = dart_co('dart_사업', c, '경쟁사', x['name'])
        if e:
            add_rel(me, 'COMPETES', e, **base(x), note=x.get('product', ''))
        else:
            miss['경쟁사'] += 1
    for x in d['contracts']:
        e = dart_co('dart_사업', c, '계약 상대', x['counterparty'])
        if e:
            add_rel(me, CONTRACT_TYPE[x['kind']], e, **base(x), kind='계약: ' + x['kind'], note=x.get('subject', ''),
                    amount=x.get('amount_text', ''), period=x.get('period_text', ''),
                    # 공급·라이선스 계약은 누가 누구에게인지 공시만으로 알 수 없을 때가 있다
                    direction_unknown=CONTRACT_TYPE[x['kind']] in ('LICENSES', 'SUPPLIES'))
        else:
            miss['계약 상대'] += 1
    for x in d['other_relations']:
        e = dart_co('dart_사업', c, '기타 관계 상대', x['counterparty'])
        if e:
            add_rel(me, 'OTHER', e, **base(x), note=x.get('role_note', ''))
        else:
            miss['기타 관계 상대'] += 1
    for b in d['brands']:
        ps = pm_dart.get((c, '브랜드', b['name']))
        for node in ps or []:
            add_rel(me, 'HAS_BRAND', node, **base(b), kind=b.get('owner_type', ''), note=b.get('product', ''))
        if not ps:
            miss['브랜드'] += 1
print('DART 사업 관계 낱개', sum(len(v) for k, v in rels.items() if v and v[0].get('source') == 'dart' and k[1] != 'OWNS'))
print('상대·제품을 못 찾아 뺀 것(그룹 이름이 아닌 애매 포함)', dict(miss))

사업보고서 접수번호 없는 회사 0
DART 사업 관계 낱개 39582
상대·제품을 못 찾아 뺀 것(그룹 이름이 아닌 애매 포함) {'고객': 2, '매입처': 1, '원재료': 1}


### 5.3 뉴스

- 뉴스에서는 **관계만** 그래프에 넣는다. 기사·사건은 노드로 두지 않고, 관계의 근거(기사 ID `A:기사ID`, 근거 문장, 발행 시각)로 남긴다. 기사 제목·링크는 출처 목록 `sources/articles.csv`에 둔다.
- 개체 → 노드: 기업은 이름 통일 결과(없으면 그룹), 제품·기술·원자재·지표는 제품 사전, 상품·브랜드는 브랜드 사전, 나머지(정책·기관·시설·프로젝트)는 `N:종류:정규화한 영어 이름`으로 만든다. 정책·기관·시설·프로젝트 노드는 관계에 쓰인 것만 남긴다.
- 뉴스까지 모은 뒤 **관계(소속·계층 제외)가 하나도 없는 노드는 뺀다**: 기업·제품·제품군·브랜드·정책·기관·시설·프로젝트. 서비스 대상 후보는 관계가 없어도 남긴다. 관계의 `product` 속성으로만 쓰인 제품도 남긴다. 남은 브랜드가 가리키는 제품·제품군, 남은 제품의 상위 제품·제품군과 후보 하위 제품은 남기고, 브랜드 → 제품은 `OF_PRODUCT`, 계층은 `SUBTYPE_OF`로 잇는다. 그래프는 매번 처음부터 다시 만들므로 나중에 관계가 생기면 그때 노드가 생긴다.

In [9]:
# ── 뉴스 관계: 기사 개체를 노드로 바꾸고 관계를 더한다. 기사는 노드가 아니라 출처 목록에 담는다 ──
NEWS = json.loads((ROOT / 'source' / 'news' / 'final' / 'news_final.json').read_text(encoding='utf-8'))['articles']
cm_news = {(r.doc, r.ref): r.entity_id for r in CM.itertuples() if r.source == 'news'}
amb_news = {(r.doc, r.ref): r.group for r in AMB.itertuples() if r.source == 'news' and r.group}
norm_en = lambda s: re.sub(r'\s+', ' ', re.sub(r'[^0-9a-z가-힣.&+-]', ' ', (s or '').lower())).strip()

def news_node(aid, e):
    """기사 개체를 노드 ID 목록으로 바꾼다. 기업은 이름 통일, 제품·브랜드는 제품 사전(여러 품목이면 여러 개), 정책·기관·시설·프로젝트는 영어 이름으로 노드를 만든다."""
    t = e['type']
    if t == '기업':
        i = cm_news.get((aid, e['id'])) or amb_news.get((aid, e['id']))
        return [i] if i else []
    if t in ('제품·기술', '원자재·지표', '상품·브랜드'):
        return pm_news.get((aid, e['id']), [])
    label, short = OTHER_LABEL[t]
    key = norm_en(e['name_en'] or e['name'])
    if not key:
        return []
    i = f'N:{short}:{key}'
    add_node(label, i, name=e['name_en'] or e['name'], name_ko=e['name'] if e['name'] != e['name_en'] else '',
             category=e.get('category', ''), country=e.get('country', ''), search_terms=e.get('search_terms', []))
    nodes[label][i]['n_news'] = nodes[label][i].get('n_news', 0) + 1
    return [i]

miss = Counter()
ARTICLES = {}                                   # 기사 ID → 기사 정보(출처 목록)
for aid, a in NEWS.items():
    ev = {e['id']: e['text'] for e in a['evidence']}
    local = {}
    for e in a['entities']:
        n = news_node(aid, e)
        if not n:
            miss['개체: ' + e['type']] += 1
            continue
        local[e['id']] = n
    for r in a['relations']:
        S, O = local.get(r['subject']), local.get(r['object'])
        if not (S and O):
            miss['관계: 끝점 없음'] += 1
            continue
        # 관계에 근거로 쓰인 기사만 출처 목록에 넣는다
        ARTICLES.setdefault(aid, dict(ref='A:' + aid, title=a['title'], source=a['source'], url=a['url'],
                                      published=a['published_utc'], article_type=a['article_type']))
        # 여러 품목 개체는 노드가 여러 개라 관계도 노드마다 만든다
        for s in S:
            for o in O:
                for pr in local.get(r['product']) or ['']:
                    add_rel(s, ROLE_TYPE[r['role']], o, source='news', article=aid, published=a['published_utc'],
                            ref='A:' + aid, date=a['published_utc'],
                            evidence=ev.get(r['evidence_id'], ''), product=pr, note=r['role_note'],
                            status=r['status'], certainty=r['certainty'], direction=r['direction'], amount=r['amount_text'],
                            period=r['period_text'])
print('관계에 근거로 쓰인 기사', len(ARTICLES))

# 관계(소속 제외)가 하나도 없는 노드는 뺀다. 서비스 대상 후보는 남긴다. 관계의 product 속성으로 쓰인 제품도 남긴다
MEMBERSHIP = {'IN_SECTOR', 'HAS_THEME', 'IN_GROUP', 'SUBTYPE_OF', 'OF_PRODUCT'}
used = {x for (st, t, en) in rels if t not in MEMBERSHIP for x in (st, en)} | {p.get('product') for ps in rels.values() for p in ps}
keep = used | {'KR:' + c for c in TARGET}
# 브랜드 → 제품: 남는 브랜드가 가리키는 제품·제품군도 남긴다(그 위 계층은 아래에서)
BRAND_P = pd.read_csv(GRAPH / 'brand_products.csv', dtype=str).fillna('')
keep |= {t for b, t in zip(BRAND_P['brand'], BRAND_P['target']) if b in keep}
# 제품 계층: 남는 제품의 상위 제품·제품군과, 남는 제품의 후보 하위 제품(검색어로 쓴다)도 남긴다
HIER = pd.read_csv(GRAPH / 'product_hierarchy.csv', dtype=str).fillna('')
up = dict(zip(HIER['child'], HIER['parent']))
for x in [x for x in keep if x in up]:
    while x in up:
        x = up[x]
        keep.add(x)
keep |= {r.child for r in HIER.itertuples() if r.child_level == '후보' and r.parent in keep}
pruned = {}
for label in ('Company', 'Product', 'ProductFamily', 'Brand', 'Policy', 'Organization', 'Facility', 'Project'):
    before = len(nodes[label])
    nodes[label] = {k: v for k, v in nodes[label].items() if k in keep}
    pruned[label] = f'{before:,} → {len(nodes[label]):,}'
print('관계가 없어 뺀 뒤 노드 수', pruned)

# 그룹 소속(IN_GROUP): 계열회사 현황에 적힌 회사 중 그래프에 남은 기업 노드에 잇는다. 근거는 그 계열회사 현황이 실린 사업보고서
#   상장 계열회사는 종목코드로, 비상장 계열회사는 DART 언급 이름(법인 표기·공백 무시)이 한 기업 노드만 가리킬 때 잇는다
aff_norm = lambda s: re.sub(r'\(주\)|㈜|주식회사|\(유\)|유한회사|[\s.,·()\-]', '', s or '').lower()
node_by_name = defaultdict(set)
for r in CM[CM['source'] != 'news'].itertuples():
    if r.entity_id:
        node_by_name[aff_norm(r.name)].add(r.entity_id)
for r in CO.itertuples():
    if r.name_ko:
        node_by_name[aff_norm(r.name_ko)].add(r.entity_id)
in_group = Counter()
for r in AFF.itertuples():
    gid = GROUP_OF.get(r.stock_code)
    if not gid:
        continue
    if r.listed == 'True':
        n = 'KR:' + r.affiliate_stock_code if r.affiliate_stock_code else None
    else:
        ids = node_by_name.get(aff_norm(r.name), set())
        n = next(iter(ids)) if len(ids) == 1 else None
    if n and n in nodes['Company'] and (n, 'IN_GROUP', gid) not in rels:
        add_rel(n, 'IN_GROUP', gid, source='dart', doc=r.stock_code, ref='D:' + r.rcept_no, date=r.base_date,
                evidence=f'{r.report} 계열회사 현황(상세): 기업집단 {r.group or gid[2:]}, {r.name_raw}')
        filing(r.rcept_no, r.stock_code, r.report, r.base_date)
        in_group['서비스 대상 후보' if n[3:] in TARGET and n.startswith('KR:') else ('후보 밖 상장사' if n.startswith('KR:') else '비상장 계열사')] += 1
print('그룹 소속', dict(in_group))

# 제품 계층(SUBTYPE_OF): 하위 → 상위. 근거 문장이 없는 분류라 사용자에게 보이는 경로에서는 업종·테마처럼 "분류"로 보여 준다
for r in HIER.itertuples():
    if r.child in nodes['Product'] and (r.parent in nodes['Product'] or r.parent in nodes['ProductFamily']):
        add_rel(r.child, 'SUBTYPE_OF', r.parent, source='llm_dictionary', kind='후보' if r.child_level == '후보' else '')
print('제품 계층', Counter('후보 → 제품' if nodes['Product'][s].get('candidate') else ('제품 → 제품군' if e.startswith('PF:') else '제품 → 상위 제품')
                        for (s, t, e) in rels if t == 'SUBTYPE_OF'))
# 브랜드 → 제품(OF_PRODUCT): LLM 3단계 결과. 근거 문장이 없는 분류다
for r in BRAND_P.itertuples():
    if r.brand in nodes['Brand'] and (r.target in nodes['Product'] or r.target in nodes['ProductFamily']):
        add_rel(r.brand, 'OF_PRODUCT', r.target, source='llm_dictionary', kind=r.target_level, note=r.how)
print('브랜드 → 제품', Counter(('→ 제품군' if e.startswith('PF:') else '→ 제품') for (s, t, e) in rels if t == 'OF_PRODUCT'),
      f"/ 이어진 브랜드 {len({s for (s, t, e) in rels if t == 'OF_PRODUCT'}):,}개(남은 브랜드 {len(nodes['Brand']):,}개)")
print('노드를 못 찾아 뺀 것', dict(miss))

관계에 근거로 쓰인 기사 1865
관계가 없어 뺀 뒤 노드 수 {'Company': '41,906 → 39,483', 'Product': '15,496 → 14,351', 'ProductFamily': '273 → 273', 'Brand': '5,596 → 5,002', 'Policy': '907 → 511', 'Organization': '1,307 → 824', 'Facility': '1,013 → 579', 'Project': '441 → 294'}


그룹 소속 {'서비스 대상 후보': 591, '후보 밖 상장사': 163, '비상장 계열사': 5815}
제품 계층 Counter({'제품 → 상위 제품': 6037, '제품 → 제품군': 5655, '후보 → 제품': 2659})
브랜드 → 제품 Counter({'→ 제품': 4343, '→ 제품군': 632}) / 이어진 브랜드 4,818개(남은 브랜드 5,002개)
노드를 못 찾아 뺀 것 {}


## 6. 합치기와 표 만들기

- 관계 끝점이 노드 표에 없으면(이름 통일 표에 없는 ID) 빼고 센다.
- 합친 관계의 속성: `sources`(dart·news·wi26·toss), `n_news`(기사 수), `articles`(최근 기사 ID, 최대 50), `filings`(공시 접수번호), `evidence`와 `evidence_refs`(근거 문장과 그 출처를 같은 순서로, 최근 것부터 최대 3, 출처마다 하나, 같은 문장은 가장 최근 출처 하나만), `kinds`, `notes`, `products`, `statuses`, `certainties`, 영향의 `plus`·`minus` 수, 지분의 `ratio`(최댓값), `first_seen`·`last_seen`(기사 발행 시각).
- 출처 목록: 근거로 쓰인 기사(`articles.csv`: 제목·매체·링크·발행 시각)와 공시(`filings.csv`: 회사·보고서·기준일·DART 링크). 그래프에는 넣지 않는다.
- 목록 속성은 CSV에 JSON 글로 적고, Neo4j에 넣을 때 목록으로 바꾼다.

In [10]:
# ── 관계 합치기: 같은 (시작, 종류, 끝)의 낱개 관계를 한 관계로. 기사 수, 출처, 근거 문장을 모은다 ──
known = {i for v in nodes.values() for i in v}
uniq = lambda xs: list(dict.fromkeys(x for x in xs if x not in (None, '', [])))

rows = defaultdict(list)
dropped = Counter()
# 끝점이 노드 표에 없는 관계는 뺀다
for (s, t, e), ps in rels.items():
    if s not in known or e not in known:
        dropped[t] += 1
        continue
    news_ps = sorted((p for p in ps if p.get('source') == 'news' and p.get('article')), key=lambda p: p['published'], reverse=True)
    row = {'start': s, 'end': e,
           'sources': uniq(p.get('source') for p in ps),
           'n_news': len({p['article'] for p in news_ps}),
           'articles': uniq(p['article'] for p in news_ps)[:MAX_ARTICLES],
           'dart_docs': uniq(p.get('doc') for p in ps if p.get('source') == 'dart'),
           'filings': uniq(p['ref'][2:] for p in ps if p.get('ref', '').startswith('D:')),
           'kinds': uniq(p.get('kind') for p in ps),
           'notes': uniq(p.get('note') for p in ps)[:5],
           'products': uniq(p.get('product') for p in ps if p.get('product') in known),
           'statuses': uniq(p.get('status') for p in ps),
           'certainties': uniq(p.get('certainty') for p in ps),
           'amounts': uniq(p.get('amount') for p in ps)[:3]}
    # 영향 관계는 +/− 방향 수를 센다
    if t == 'AFFECTS':
        row['plus'] = sum(p.get('direction') == '+' for p in ps)
        row['minus'] = sum(p.get('direction') == '−' for p in ps)
    if t == 'OWNS':
        rs = [p['ratio'] for p in ps if p.get('ratio') is not None]
        row['ratio'] = max(rs) if rs else None
    ev, seen, seen_text = [], set(), set()   # 근거와 출처: 최근 것부터, 출처마다 하나, 같은 문장은 한 번만(가장 최근 출처)
    for p in sorted((p for p in ps if p.get('ref') and p.get('evidence')), key=lambda p: p.get('date') or '', reverse=True):
        text_key = re.sub(r'\s+', ' ', p['evidence']).strip()
        if p['ref'] not in seen and text_key not in seen_text:
            seen.add(p['ref']); seen_text.add(text_key); ev.append((p['evidence'], p['ref']))
    if ev:
        row['evidence'] = [x for x, _ in ev[:MAX_EVIDENCE]]
        row['evidence_refs'] = [r for _, r in ev[:MAX_EVIDENCE]]
    if any(p.get('direction_unknown') for p in ps):
        row['direction_unknown'] = True
    if news_ps:
        row['first_seen'], row['last_seen'] = news_ps[-1]['published'], news_ps[0]['published']
    rows[t].append(row)
print('끝점이 노드 표에 없어 뺀 관계', dict(dropped))
summary = pd.DataFrame([(t, len(v), sum('news' in r['sources'] for r in v), sum('dart' in r['sources'] for r in v),
                         sum('news' in r['sources'] and 'dart' in r['sources'] for r in v)) for t, v in rows.items()],
                       columns=['관계', '수', '뉴스', 'DART', '둘 다']).sort_values('수', ascending=False)
display(summary)
display(pd.DataFrame([(k, len(v)) for k, v in nodes.items()], columns=['노드', '수']).sort_values('수', ascending=False))

끝점이 노드 표에 없어 뺀 관계 {}


,관계,수,뉴스,DART,둘 다
2,OWNS,39777,0,39777,0
3,PRODUCES,15688,1983,13716,11
21,SUBTYPE_OF,14351,0,0,0
20,IN_GROUP,6569,0,6569,0
7,SUPPLIES,6465,390,6078,3
4,BUYS,6074,554,5520,0
9,HAS_BRAND,4998,0,4998,0
22,OF_PRODUCT,4975,0,0,0
11,OTHER,2252,236,2016,0
1,HAS_THEME,2153,0,0,0


,노드,수
0,Company,39483
4,Product,14351
6,Brand,5002
7,Organization,824
9,Facility,579
8,Policy,511
10,Project,294
5,ProductFamily,273
1,Group,240
3,Theme,125


In [11]:
# ── 저장: 노드 표·관계 표(Neo4j용 CSV)와 출처 목록(기사·공시) ──
def to_frame(records):
    """목록 속성은 JSON 문자열로 바꿔 CSV로 쓸 수 있게 한다."""
    df = pd.DataFrame(records)
    for c in df.columns:
        if df[c].map(lambda x: isinstance(x, list)).any():
            df[c] = df[c].map(lambda x: json.dumps(x if isinstance(x, list) else [], ensure_ascii=False))
    return df

def write_if_changed(path, df):
    """내용이 바뀌었을 때만 CSV를 쓴다(iCloud 동기화 사본이 생기지 않게)."""
    content = '﻿' + df.to_csv(index=False)
    if path.exists() and path.read_text(encoding='utf-8') == content:
        return False
    path.write_text(content, encoding='utf-8')
    return True

written = []
for label, v in nodes.items():
    written.append(write_if_changed(OUT / f'nodes_{label}.csv', to_frame(sorted(v.values(), key=lambda x: x['id']))))
for t, v in rows.items():
    written.append(write_if_changed(OUT / f'rels_{t}.csv', to_frame(sorted(v, key=lambda x: (x['start'], x['end'])))))
written.append(write_if_changed(SRC / 'articles.csv', pd.DataFrame(sorted(ARTICLES.values(), key=lambda x: x['ref']))))
written.append(write_if_changed(SRC / 'filings.csv', pd.DataFrame(sorted(FILINGS.values(), key=lambda x: x['ref']))))
print('출처 목록: 기사', len(ARTICLES), '/ 공시', len(FILINGS))
stale = [p.name for p in OUT.glob('*.csv') if p.stem.split('_', 1)[1] not in set(nodes) | set(rows)]
for name in stale:                      # 이번에 안 만든 표(이전 실행의 남은 것)는 지운다
    (OUT / name).unlink()
print('표', len(written), '개 중 바뀐 것', sum(written), '/ 지운 옛 표', stale)

출처 목록: 기사 1865 / 공시 2156
표 36 개 중 바뀐 것 0 / 지운 옛 표 []


## 7. Neo4j에 넣기

- 접속 정보는 `.env`(`NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`)에서 읽는다. 비밀번호는 출력하지 않는다.
- **통째로 다시 넣는다**: 노드·관계를 모두 지우고, `Node.id` 고유 제약을 걸고, 위 표를 넣는다.
- 노드는 공통 라벨 `Node`와 종류 라벨(`Company` 등)을 함께 단다. 빈 값은 넣지 않는다.

In [12]:
# ── Neo4j에 넣기: 모두 지우고 노드 → 관계 순서로 UNWIND 묶음 CREATE ──
from neo4j import GraphDatabase

def env_value(key):
    """.env에서 값을 읽는다. 비밀번호는 노트북에 쓰지 않는다."""
    for line in (ROOT.parent / '.env').read_text(encoding='utf-8-sig').splitlines():
        k, _, v = line.partition('=')
        if k.strip() == key:
            return v.strip().strip('"\'')
    raise KeyError(f'.env에 {key}가 없다')

def clean(d):
    """빈 값(None, NaN, '', [])을 뺀다. Neo4j는 null 속성을 둘 수 없다."""
    out = {}
    for k, v in d.items():
        if v is None or (isinstance(v, float) and pd.isna(v)) or v == '' or v == []:
            continue
        out[k] = v
    return out

def chunks(xs, n=BATCH):
    """목록을 n개씩 나눈다."""
    for i in range(0, len(xs), n):
        yield xs[i:i + n]

if LOAD:
    driver = GraphDatabase.driver(env_value('NEO4J_URI'), auth=(env_value('NEO4J_USER'), env_value('NEO4J_PASSWORD')))
    try:
        driver.verify_connectivity()
    except Exception as e:
        raise SystemExit(f'Neo4j에 접속하지 못했다({type(e).__name__}). Docker Desktop과 neo4j-graph 컨테이너가 켜져 있는지 확인한다.')
    with driver.session() as ses:
        # 전체 다시 넣기: 트랜잭션을 나눠 모두 지운다
        ses.run('MATCH (n) CALL (n) { DETACH DELETE n } IN TRANSACTIONS OF 10000 ROWS').consume()
        # 모든 노드에 공통 라벨 Node와 고유한 id
        ses.run('CREATE CONSTRAINT node_id IF NOT EXISTS FOR (n:Node) REQUIRE n.id IS UNIQUE').consume()
        for label, v in nodes.items():
            for part in chunks([clean(x) for x in v.values()]):
                ses.run(f'UNWIND $rows AS r CREATE (n:Node:{label}) SET n = r', rows=part).consume()
        for t, v in rows.items():
            data = [{'s': r['start'], 'e': r['end'], 'p': clean({k: x for k, x in r.items() if k not in ('start', 'end')})} for r in v]
            for part in chunks(data):
                ses.run(f'UNWIND $rows AS r MATCH (a:Node {{id: r.s}}) MATCH (b:Node {{id: r.e}}) CREATE (a)-[x:{t}]->(b) SET x = r.p',
                        rows=part).consume()
        n_nodes = ses.run('MATCH (n) RETURN count(n) AS c').single()['c']
        n_rels = ses.run('MATCH ()-[r]->() RETURN count(r) AS c').single()['c']
    print('Neo4j 노드', n_nodes, '(표', sum(len(v) for v in nodes.values()), ') / 관계', n_rels, '(표', sum(len(v) for v in rows.values()), ')')
else:
    print('LOAD=False라 Neo4j에 넣지 않았다.')

Neo4j 노드 61708 (표 61708 ) / 관계 111954 (표 111954 )


## 8. 확인

Neo4j에서 몇 가지를 물어 본다. 브라우저(http://localhost:7474)에서도 같은 질의를 쓸 수 있다.

In [13]:
# ── 확인: 테마와 업종, 제품을 거치는 경로, 사용자에게 보여 줄 근거와 출처 ──
def q(cypher, **kw):
    """Cypher를 실행해 표로 돌려준다."""
    with driver.session() as ses:
        return pd.DataFrame([r.data() for r in ses.run(cypher, **kw)])

if LOAD:
    print('① 테마 방산 종목과 WI26 업종')
    display(q("""MATCH (c:Company)-[:HAS_THEME]->(:Theme {name:'방산'}) OPTIONAL MATCH (c)-[:IN_SECTOR]->(s:Sector)
                 RETURN s.name AS 업종, count(c) AS 종목수, collect(c.name)[..5] AS 예 ORDER BY 종목수 DESC"""))
    print('② 뉴스와 DART를 함께 거치는 상장사 → 제품 (DRAM과 그 하위 제품: HBM 등)')
    display(q("""MATCH (p:Product)-[:SUBTYPE_OF*0..3]->(:Product {id: 'P:dram'})
                 MATCH (x)-[r:PRODUCES|BUYS]->(p) RETURN p.name AS 제품, type(r) AS 관계, x.name AS 주체,
                 coalesce(x.listed, false) AS 상장사, r.sources AS 출처, r.n_news AS 기사수 ORDER BY 기사수 DESC LIMIT 15"""))
    print('②-2 제품 계층(분류): HBM에서 제품군까지, 후보 하위 제품')
    display(q("""MATCH path = (:Product {id: 'P:high-bandwidth memory'})-[:SUBTYPE_OF*]->(:ProductFamily)
                 RETURN [n IN nodes(path) | n.name] AS 분류"""))
    display(q("""MATCH (c:Product {candidate: true})-[:SUBTYPE_OF]->(p:Product)-[:SUBTYPE_OF*0..3]->(:ProductFamily {id: 'PF:memory_chips'})
                 RETURN p.name AS 상위제품, c.name AS 후보, c.search_terms AS 검색어 LIMIT 8"""))
    print('②-3 브랜드 → 제품(분류) → 국내 생산 기업: Zepbound')
    display(q("""MATCH (b:Brand {id: 'B:zepbound'})-[:OF_PRODUCT]->(p)<-[r:PRODUCES]-(c:Company)
                 RETURN b.name AS 브랜드, p.name AS 제품, c.name AS 기업, coalesce(c.listed, false) AS 상장사, r.sources AS 출처 LIMIT 10"""))
    print('③ 사용자에게 보여 줄 근거: SK하이닉스와 해외 기업의 관계와 출처')
    src = pd.concat([pd.read_csv(SRC / 'articles.csv', dtype=str).assign(출처=lambda d: d['source'] + ' ' + d['published'].str[:10] + ' | ' + d['title']),
                     pd.read_csv(SRC / 'filings.csv', dtype=str).assign(출처=lambda d: 'DART ' + d['corp_name'] + ' ' + d['report'])])[['ref', '출처']]
    ex = q("""MATCH (k:Company {name:'SK하이닉스'})-[r]-(x:Company) WHERE x.id STARTS WITH 'EN:' AND r.evidence IS NOT NULL
             RETURN type(r) AS 관계, CASE WHEN startNode(r) = k THEN '→' ELSE '←' END AS 방향, x.name AS 상대,
                    r.evidence[0] AS 근거, r.evidence_refs[0] AS ref, r.n_news AS 기사수 ORDER BY 기사수 DESC LIMIT 10""")
    display(ex.merge(src, on='ref', how='left').drop(columns='ref'))
    print('④ 업종 소속이 없는 서비스 대상 후보(있으면 확인)')
    display(q("""MATCH (c:Company {target:true}) WHERE NOT (c)-[:IN_SECTOR]->() RETURN count(c) AS 수"""))
    driver.close()

① 테마 방산 종목과 WI26 업종


,업종,종목수,예
0,"상사,자본재",14,"[퍼스텍, 두산, 현대로템, LIG디펜스앤에어로스페이스, 아이쓰리시스템]"
1,자동차,6,"[현대차, 평화산업, SNT모티브, SNT다이내믹스, 평화홀딩스]"
2,IT하드웨어,3,"[파이버프로, 인텔리안테크, RFHIC]"
3,조선,2,"[한화오션, STX엔진]"
4,비철금속,2,"[풍산홀딩스, 풍산]"
5,반도체,2,"[뉴파워프라즈마, 덕산하이메탈]"
6,IT가전,1,[비츠로셀]
7,건설,1,[HJ중공업]
8,건강관리,1,[한컴라이프케어]
9,에너지,1,[SK오션플랜트]


② 뉴스와 DART를 함께 거치는 상장사 → 제품 (DRAM과 그 하위 제품: HBM 등)


,제품,관계,주체,상장사,출처,기사수
0,고대역폭 메모리,PRODUCES,삼성전자,True,"[dart, news]",2
1,고대역폭 메모리,PRODUCES,SK하이닉스,True,"[dart, news]",2
2,저전력 D램,PRODUCES,SK하이닉스,True,[news],1
3,고대역폭 메모리,PRODUCES,마이크론,False,[news],1
4,DDR 동기식 D램,PRODUCES,한양디지텍,True,[dart],0
5,DDR 동기식 D램,PRODUCES,제주반도체,True,[dart],0
6,디램,PRODUCES,제주반도체,True,[dart],0
7,디램,PRODUCES,삼성전자,True,[dart],0
8,DDR 동기식 D램,PRODUCES,삼성전자,True,[dart],0
9,디램,PRODUCES,SK하이닉스,True,[dart],0


②-2 제품 계층(분류): HBM에서 제품군까지, 후보 하위 제품


,분류
0,"[고대역폭 메모리, 디램, 랜덤 액세스 메모리, 메모리 반도체]"


... (3 outputs left out)

③ 사용자에게 보여 줄 근거: SK하이닉스와 해외 기업의 관계와 출처


,관계,방향,상대,근거,기사수,출처
0,INVESTS,→,Kioxia Holdings Corporation,The company also announced a special dividend ...,2,CNBC 2026-10-28 | Stocks making the biggest mo...
1,SUPPLIES,→,Nvidia,"Nvidia, No. 16 on the 2026 Fortune 500, relies...",1,Fortune 2026-11-10 | Memory chip CEO warns 202...
2,OWNS,→,"CHONGQING HAPPYNARAE Co., Ltd",사업보고서 (2025.12) 종속회사 표: CHONGQING HAPPYNARAE C...,0,DART SK하이닉스 사업보고서 (2025.12)
3,OWNS,→,FMC,반기보고서 (2026.06) 출자 대상 표: FMC,0,DART SK하이닉스 반기보고서 (2026.06)
4,OWNS,→,"GigaIO Networks, Inc.","반기보고서 (2026.06) 출자 대상 표: GigaIO Networks, Inc.",0,DART SK하이닉스 반기보고서 (2026.06)
5,OWNS,→,HappyNarae America LLC,사업보고서 (2025.12) 종속회사 표: HappyNarae America LLC,0,DART SK하이닉스 사업보고서 (2025.12)
6,OWNS,→,Intel NDTM US LLC,사업보고서 (2025.12) 종속회사 표: Intel NDTM US LLC,0,DART SK하이닉스 사업보고서 (2025.12)
7,OWNS,→,HappyNarae Hungary Kft,사업보고서 (2025.12) 종속회사 표: HappyNarae Hungary Kft,0,DART SK하이닉스 사업보고서 (2025.12)
8,OWNS,→,"MEMS Drive, Inc.","반기보고서 (2026.06) 출자 대상 표: MEMS Drive, Inc.",0,DART SK하이닉스 반기보고서 (2026.06)
9,OWNS,→,"FemtoMetrix, Inc.","반기보고서 (2026.06) 출자 대상 표: FemtoMetrix, Inc.",0,DART SK하이닉스 반기보고서 (2026.06)


④ 업종 소속이 없는 서비스 대상 후보(있으면 확인)


,수
0,0
